In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
TEMP_CSV   = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV  = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")

pd.set_option("display.max_columns", 50)
print("경로 확인:", os.path.exists(TEMP_CSV), os.path.exists(EVENT_CSV), os.path.exists(REPAIR_CSV))

In [ ]:
df_temp = pd.read_csv(TEMP_CSV, encoding="utf-8") # T-CR1-CAL1_온도.csv
df_event = pd.read_csv(EVENT_CSV, encoding="utf-8") # G-02_조업이벤트.csv
df_repair = pd.read_csv(REPAIR_CSV, encoding="utf-8") # G-01_정기수리캘린더.csv

display(df_temp.head(5))

In [ ]:
df_event.head(5)

In [ ]:
df_repair.head(5)

In [ ]:
print(df_temp['MEAS_DT'].dtype)
df_temp['MEAS_DT'] = pd.to_datetime(df_temp['MEAS_DT'])
print(df_temp['MEAS_DT'].dtype)

In [ ]:
df_event["EVT_DT"] = pd.to_datetime(df_event["EVT_DT"])
print(df_event["EVT_DT"].dtype)

In [ ]:
df_repair["STRT_DT"] = pd.to_datetime(df_repair["STRT_DT"])
df_repair["END_DT"] = pd.to_datetime(df_repair["END_DT"])
print(df_repair[["STRT_DT", "END_DT"]])

print(df_repair[["STRT_DT", "END_DT"]].dtypes)

---

In [ ]:
print(df_event[df_event['EVT_TYPE'] == '강종변경']['BEF_VAL'].unique())
print("=" * 50)
print(df_event[df_event['EVT_TYPE'] == '강종변경']['AFT_VAL'].unique())
print("="*50)
print(df_temp['LOT_NO'].unique())
print("LOT_NO의 unique값 개수:", len(df_temp['LOT_NO'].unique()))

---
## **의문** 1. 강종 수와 LOT
- `G-02_조업이벤트.csv`의 `AFT_VAL(=BEF_VAL)`의 `unique`값은
    - ['SGCC', 'SPFH590', 'SPCD', 'SPHC', 'SPCEN', 'SPCE', 'SPCC', 'DP590'] => 8가지

- `T-CR1-CAR1_온도.csv`의 `LOT_NO`의 `unique`값은
    - [240001 240024 240037 240038 240044 240058 240061 240069 240091 240108
 240114 240117 240121 240133] => 14가지

> 강종 변경이 곧 LOT_NO가 바뀌는 걸 의미하지 않는다는 걸 추측해볼 수 있음  
> 그래프로 확인해봅시다  

### **(1)** **BEF_VAL -> AFT_VAL** 체인 연결 확인

In [ ]:
grade_df = df_event[df_event['EVT_TYPE'] == '강종변경'].sort_values("EVT_DT").reset_index(drop=True)
print("강종변경 이벤트 수", len(grade_df))
chain_ok = grade_df["AFT_VAL"].shift(1) == grade_df["BEF_VAL"]
chain_ok.iloc[0] = True # 첫 행은 비교 대상이 없으니 통과
print("체인이 끊긴 지점 수:", (~chain_ok).sum(), "/", len(grade_df))
display(grade_df[~chain_ok][["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL"]])

> 모든 건에서의 강종 변경은 이전 조업이벤트에서 변경된 값임을 알 수 있음  

### **(2) 온도.csv의 바뀐 시점과 조업이벤트.csv가 강종 변경으로 바뀐 시점 비교**
- 강종 변경의 이유에는 이상이 없으니, 온도.csv의 `MEAS_DT`와 조업이벤트.csv의 `EVT_DT`가 비슷한 시기인지 확인해봅시다

In [ ]:
lot_change_mask = df_temp['LOT_NO'] != df_temp['LOT_NO'].shift(1)
lot_changes = df_temp.loc[lot_change_mask, ["MEAS_DT", "LOT_NO"]].reset_index(drop=True)

print("LOT_NO 변경 횟수:", len(lot_changes))
display(lot_changes)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 3))

ax.eventplot(grade_df['EVT_DT'], lineoffsets=1, linelengths=0.8, colors='tab:blue')
ax.eventplot(lot_changes['MEAS_DT'], lineoffsets=2, linelengths=0.8, colors='tab:red')

ax.set_yticks([1,2])
ax.set_yticklabels(
    [f"강종변경\n({len(grade_df)}건)", f"LOT 변경\n({len(lot_changes)}건)"]
)

ax.set_xlabel('날짜')
ax.set_title('강종변경 vs 온도 LOT_NO 변경 시점')
plt.show()

### **(3) 강종 8종류가 한 사이클이 돌아야 LOT_NO가 바뀌는지 확인**
- 위 그래프로는 적절하게 추측해볼 변경점이 명확하지 않으니, 사이클이어야 하는지 확인해봅시다

In [ ]:
# 강종변경 이벤트마다 그 시점의 LOT_NO 붙이기 (EVT_DT 직전의 LOT 변경 시점을 찾아 연결)
grade_lot = pd.merge_asof(grade_df, lot_changes, left_on="EVT_DT", right_on="MEAS_DT")

# LOT별로 몇 종류의 강종이 나왔는지 세기 (강종변경이 없는 LOT은 0)
lot_grade_count = grade_lot.groupby("LOT_NO")["AFT_VAL"].nunique()
lot_grade_count = lot_grade_count.reindex(lot_changes["LOT_NO"], fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
lot_grade_count.plot.bar(ax=ax, color="tab:blue")
ax.axhline(8, color="gray", linestyle="--", label="전체 강종 수 (8)")
ax.set_xlabel("LOT_NO")
ax.set_ylabel("LOT 안에서 나온 강종 수")
ax.set_title("LOT별 강종 종류 수 (8종을 다 돌아야 LOT_NO가 바뀌는지 확인)")
ax.legend()
plt.tight_layout()
plt.show()

> 14개 LOT 가운데 8종을 모두 거친 LOT은 240091 하나뿐이고, 1~2종만 나온 LOT도 있습니다. 따라서 "강종 8종을 한 바퀴 돌아야 LOT_NO가 바뀐다"는 가설은 맞지 않는 듯 함

In [ ]:
# 보충 분석: LOT_NO  변경 시점이 강종 변경 시점과 정확히 일치하는가?
ls = []

for _, row in lot_changes.iterrows():
    diffs = (grade_df['EVT_DT'] - row['MEAS_DT']).abs()
    idx = diffs.idxmin()
    ls.append(
        {
            "LOT_NO": row["LOT_NO"],
            "LOT_변경시각": row["MEAS_DT"],
            "가장_가까운_강종변경_시각": grade_df.loc[idx, "EVT_DT"],
            "시간차": diffs.min(),
            "BEF_VAL": grade_df.loc[idx, "BEF_VAL"],
        
            "AFT_VAL": grade_df.loc[idx, "AFT_VAL"],
        }
    
    )

lot_vs_grade = pd.DataFrame(ls)
display(lot_vs_grade)

### **(4) LOT_NO 한 구간 안에서 강종이 몇 번 바뀌는지 확인**

In [ ]:
lot_bounds = lot_changes['MEAS_DT'].tolist() + [df_temp['MEAS_DT'].max() + pd.Timedelta(seconds=1)]

grade_counts = []
for i in range(len(lot_changes)):
    start, end = lot_bounds[i], lot_bounds[i + 1]
    n_greade_events = grade_df['EVT_DT'].between(start,end, inclusive='left').sum()
    grade_counts.append(n_greade_events)

lot_changes["강종변경_횟수"] = grade_counts
display(lot_changes)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    lot_changes["LOT_NO"].astype(str), lot_changes["강종변경_횟수"], color="tab:blue"
)
ax.set_xlabel("LOT_NO")
ax.set_ylabel("구간 내 강종변경 횟수")
ax.set_title("LOT_NO 구간별 강종변경 발생 횟수")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### **(5) 강종이 바뀌고 나서 다음 변경까지 얼마나 유지되는지 확인**

In [ ]:
grade_df["유지시간"] = grade_df["EVT_DT"].shift(-1) - grade_df["EVT_DT"]

display(grade_df.groupby("AFT_VAL")["유지시간"].describe())

fig, ax = plt.subplots(figsize=(8, 4))
grade_df["유지시간"].dt.total_seconds().div(3600).dropna().hist(bins=30, ax=ax)
ax.set_xlabel("유지 시간 (시간)")
ax.set_ylabel("빈도")
ax.set_title("강종 변경 후 다음 변경까지 유지 시간 분포")
plt.tight_layout()
plt.show()

### **(6) 온도 센서 실측값에 LOT 경계 / 강종 변경 이벤트 겹쳐서 확인**

In [ ]:
sensor_col = "TC-OUT1"
temp_hourly = df_temp.set_index("MEAS_DT")[sensor_col].resample("1h").mean()

fig, ax = plt.subplots(figsize=(16, 4))
temp_hourly.plot(ax=ax, color="gray", linewidth=0.8)

for dt in grade_df["EVT_DT"]:
    ax.axvline(dt, color="tab:blue", alpha=0.15, linewidth=0.8)
for dt in lot_changes["MEAS_DT"]:
    ax.axvline(dt, color="tab:orange", alpha=0.8, linewidth=1.5)

ax.set_title(f"{sensor_col} 시간별 평균 (파랑=강종변경, 주황=LOT_NO 변경)")
ax.set_ylabel(sensor_col)
plt.tight_layout()
plt.show()